# Aula 11 — Laboratório: Curvas de aprendizado

**Consultoria Especializada em Ciência de Dados 1**

Na Aula 10 a curva de validação variou **um hiperparâmetro** com a base parada, e respondeu
"qual valor usar?". Hoje a gente tranca o modelo e varia o **tamanho do treino**, para responder
outra pergunta: **mais dados ajudariam?**

A pergunta não é técnica, é de dinheiro — rotular mil casos custa. O que a curva faz é trocar
achismo por evidência.

Mesma base clínica da Aula 10: prever quais pacientes são de **alto risco metabólico**.

## Parte 0 — Ambiente, dados e a função de desenho

Três pipelines, todos com o `StandardScaler` antes do modelo: dentro da validação cruzada a
padronização é ajustada **só na parte de treino** de cada dobra. A métrica é o **ROC-AUC** —
a base tem 15% de positivos, e a Aula 10 já mostrou o ponto cego da acurácia aqui.

`plota_aprendizado` desenha as duas linhas com a faixa de ± 1 desvio entre as dobras.
Rode as duas células abaixo sem alterar nada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (learning_curve, StratifiedKFold,
                                     cross_val_score, train_test_split)

pac = pd.read_csv("datasets/1_pacientes.csv")
pac["alto_risco"] = (pac["risco_metabolico"] == "alto").astype(int)
feat = [c for c in pac.columns if c not in ["id", "risco_metabolico", "alto_risco", "sexo"]]
X, y = pac[feat], pac["alto_risco"]

logistica = Pipeline([("scaler", StandardScaler()),
                      ("clf", LogisticRegression(max_iter=1000))])
arvore15  = Pipeline([("scaler", StandardScaler()),
                      ("clf", DecisionTreeClassifier(max_depth=15, random_state=42))])
toco      = Pipeline([("scaler", StandardScaler()),
                      ("clf", DecisionTreeClassifier(max_depth=1, random_state=42))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # as MESMAS dobras sempre
frac = np.linspace(0.1, 1.0, 8)                                   # 8 tamanhos de treino

print("pacientes:", len(y), "| variáveis:", len(feat))
print("alto risco:", int(y.sum()), f"({y.mean():.1%})")
print("frações do treino:", np.round(frac, 3))

In [ ]:
def plota_aprendizado(tam, tr, va, titulo):
    """tam, tr e va são o que learning_curve devolve: tamanhos, treino, validação."""
    tr_m, tr_s = tr.mean(axis=1), tr.std(axis=1)
    va_m, va_s = va.mean(axis=1), va.std(axis=1)
    plt.figure(figsize=(8, 4.2))
    plt.plot(tam, tr_m, "o-", color="#0F766E", label="treino")
    plt.fill_between(tam, tr_m - tr_s, tr_m + tr_s, alpha=.15, color="#0F766E")
    plt.plot(tam, va_m, "o-", color="#B45309", label="validação")
    plt.fill_between(tam, va_m - va_s, va_m + va_s, alpha=.15, color="#B45309")
    plt.xlabel("exemplos no treino"); plt.ylabel("ROC-AUC")
    plt.title(titulo); plt.legend(); plt.grid(alpha=.25)
    plt.tight_layout(); plt.show()
    return va_m[-1] - va_m[-2]        # a inclinação final: o ganho do último passo

## Parte 1 — A primeira curva de aprendizado

A `learning_curve` recebe o Pipeline inteiro e, para **cada** tamanho de treino, faz uma validação
cruzada completa. Ela devolve **três** coisas: o vetor de tamanhos e duas matrizes
(tamanho × dobra), uma de treino e outra de validação.

Repare na diferença para a Aula 10: lá o `param_range` listava valores de um hiperparâmetro; aqui
o `train_sizes` lista **frações da base de treino**.

### Exemplo — três tamanhos, para ver o formato da saída

Antes de varrer tudo, olhe o que a função devolve com apenas três tamanhos.

In [ ]:
tam3, tr3, va3 = learning_curve(
    logistica, X, y,
    train_sizes=[0.1, 0.5, 1.0],
    cv=cv, scoring="roc_auc")

print("tamanhos:", tam3, " | formato das matrizes:", tr3.shape, "-> 3 tamanhos x 5 dobras")
for t, a, b in zip(tam3, tr3.mean(axis=1), va3.mean(axis=1)):
    print(f"{t:4d} exemplos   treino {a:.4f}   validação {b:.4f}   gap {a - b:.4f}")

### Exercício 1 — a curva completa da logística

Refaça a chamada com os **8 tamanhos** do vetor `frac` e desenhe o resultado com
`plota_aprendizado`. Guarde as três saídas em `tam`, `tr` e `va`.

In [ ]:
# TODO: gere a curva de aprendizado da 'logistica' com train_sizes=frac
tam, tr, va = ____

m_tr, m_va, s_va = tr.mean(axis=1), va.mean(axis=1), va.std(axis=1)
inclinacao = plota_aprendizado(tam, tr, va, "logística — curva de aprendizado")
print("ganho do último passo (inclinação final):", round(inclinacao, 4))

## Parte 2 — Ler a curva: as duas linhas e o gap

Com mais exemplos, o treino **cai** um pouco (fica mais difícil decorar tudo) e a validação
**sobe** (o modelo generaliza melhor). O **gap** = treino − validação tende a encolher.

E a regra que não muda: na hora de decidir, você lê a **linha de validação**. A de treino serve
para uma coisa só — comparada com a outra, ela dá o gap.

### Exemplo — a tabela dos três tamanhos da Parte 1

In [ ]:
exemplo = pd.DataFrame({"exemplos":  tam3,
                        "treino":    tr3.mean(axis=1).round(4),
                        "validação": va3.mean(axis=1).round(4)})
exemplo["gap"] = (exemplo["treino"] - exemplo["validação"]).round(4)
exemplo

### Exercício 2 — a tabela completa e o gap

Monte um `DataFrame` com os 8 tamanhos e as colunas `treino`, `validação`, `desvio` e `gap`.
Acrescente uma coluna `ganho` com a diferença de validação para a linha anterior
(use `.diff()`).

Responda ao final: o gap encolhe? A validação ainda sobe no último passo?

In [ ]:
tabela = pd.DataFrame({"exemplos":  tam,
                       "treino":    m_tr.round(4),
                       "validação": m_va.round(4),
                       "desvio":    s_va.round(4)})

# TODO: crie a coluna 'gap' (treino - validação) e a coluna 'ganho' (diferença de validação)
tabela["gap"]   = ____
tabela["ganho"] = ____
tabela

**Resposta:**

## Parte 3 — As duas assinaturas: viés alto e variância alta

A forma da curva diz qual é o gargalo:

* **Viés alto (subajuste)** — as duas linhas se encontram num patamar **baixo**, com gap pequeno.
  Falta capacidade, não dado.
* **Variância alta (sobreajuste)** — treino **alto**, validação bem abaixo, gap grande.
  O modelo decora.

Os dois extremos estão prontos no Parte 0: a `arvore15` (profundidade 15) e o `toco`
(profundidade 1).

### Exemplo — a assinatura da árvore profunda

In [ ]:
tam_a, tr_a, va_a = learning_curve(arvore15, X, y, train_sizes=frac,
                                   cv=cv, scoring="roc_auc")
print("treino no maior tamanho   :", round(tr_a.mean(axis=1)[-1], 4))
print("validação no maior tamanho:", round(va_a.mean(axis=1)[-1], 4))
print("gap                       :", round(tr_a.mean(axis=1)[-1] - va_a.mean(axis=1)[-1], 4))

### Exercício 3 — os dois extremos, lado a lado

Gere a curva do `toco` e desenhe as duas (árvore 15 e toco) com `plota_aprendizado`. Depois
imprima, para os **três** modelos, o treino, a validação e o gap no maior tamanho.

Responda: qual é viés alto e qual é variância alta? E em qual dos dois **mais dados** ajudaria?

In [ ]:
plota_aprendizado(tam_a, tr_a, va_a, "árvore profundidade 15")

# TODO: gere a curva do 'toco' e desenhe
tam_t, tr_t, va_t = ____
plota_aprendizado(tam_t, tr_t, va_t, "árvore profundidade 1 (toco)")

for nome, t, v in [("logística ", tr, va), ("árvore 15 ", tr_a, va_a), ("toco      ", tr_t, va_t)]:
    a, b = t.mean(axis=1)[-1], v.mean(axis=1)[-1]
    print(f"{nome} treino {a:.4f}   validação {b:.4f}   gap {a - b:.4f}")

**Resposta:**

## Parte 4 — A decisão: vale coletar mais?

A régua é a **inclinação final** da linha de validação. Se ainda sobe, coletar provavelmente
rende; se achatou num platô, o ganho será pequeno e caro.

E "pequeno" só significa alguma coisa comparado com o **desvio**: um ganho menor que a variação
entre as dobras não é ganho demonstrado.

### Exemplo — o ganho do último passo contra o desvio daquele ponto

In [ ]:
print(f"validação no penúltimo tamanho ({tam[-2]}): {m_va[-2]:.4f}")
print(f"validação no último tamanho    ({tam[-1]}): {m_va[-1]:.4f} ± {s_va[-1]:.4f}")
print(f"ganho do passo : {m_va[-1] - m_va[-2]:+.4f}")
print(f"desvio do ponto: {s_va[-1]:.4f}   -> o ganho cabe dentro do desvio?",
      (m_va[-1] - m_va[-2]) < s_va[-1])

### Exercício 4 — a resposta ao cliente

O cliente pergunta: *"vale rotular mais mil pacientes?"*. Responda com números.

Monte uma tabela com o **ganho por passo** e o **ganho por 100 exemplos novos** (ganho dividido
pelo número de exemplos acrescentados, vezes 100). Depois escreva a resposta em uma frase, citando
a inclinação final e o desvio.

In [ ]:
ganho = np.diff(m_va)
novos = np.diff(tam)

# TODO: ganho por 100 exemplos novos
por100 = ____

for i in range(len(ganho)):
    print(f"de {tam[i]:3d} para {tam[i+1]:3d} (+{novos[i]:3d})   "
          f"ganho {ganho[i]:+.4f}   por 100 exemplos {por100[i]:+.4f}")

**Resposta:**

## Parte 5 — Base pequena: onde a leitura muda

Com pouco dado você está na parte **íngreme** da curva, e é ali que cada lote novo rende mais.
A curva fica mais útil, não menos. Mas a leitura exige dois cuidados: o primeiro ponto é ruído,
e a comparação tem de ser entre **faixas**, não entre médias.

### Exemplo — o primeiro ponto com 200 linhas

Vamos cortar a base para 200 pacientes e pedir a curva começando em 10%. O primeiro ponto treina
com 16 exemplos — e repare no que acontece.

In [ ]:
X_peq, _, y_peq, _ = train_test_split(X, y, train_size=200, stratify=y, random_state=42)
print("base pequena:", len(y_peq), "pacientes |", int(y_peq.sum()), "de alto risco")

tam_p, tr_p, va_p = learning_curve(logistica, X_peq, y_peq, train_sizes=frac,
                                   cv=cv, scoring="roc_auc")
print("\ntamanhos:", tam_p)
print("validação:", np.round(va_p.mean(axis=1), 4))
print("\nO primeiro ponto treina com", tam_p[0], "exemplos.")
print("Olhe o aviso que o scikit-learn emite e o valor que saiu nesse ponto.")

### Exercício 5 — a faixa, não a média

Refaça a curva da base pequena começando em **25%** (`np.linspace(0.25, 1.0, 8)`) e imprima,
para cada tamanho, a média da validação e a **faixa** de ± 1 desvio.

Responda: entre o primeiro e o último tamanho, as faixas se sobrepõem? Dá para afirmar que a
curva subiu?

In [ ]:
frac_p = np.linspace(0.25, 1.0, 8)

# TODO: gere a curva da base pequena com train_sizes=frac_p
tam_q, tr_q, va_q = ____

mq, sq = va_q.mean(axis=1), va_q.std(axis=1)
for t, m, s in zip(tam_q, mq, sq):
    print(f"{t:4d} exemplos   validação {m:.4f} ± {s:.4f}   faixa [{m - s:.3f} ; {m + s:.3f}]")

**Resposta:**

## Parte 6 — Quanto dado basta, e quanto custa descobrir

Esta parte responde a uma dúvida que veio no formulário: se nem sempre é preciso treinar com a
base inteira, **como saber que uma amostra basta** — e como evitar que descobrir isso custe o
mesmo que treinar com tudo?

A primeira pergunta a curva responde de graça: o tamanho suficiente é o **menor** cuja faixa já
alcança o desempenho do maior. A segunda tem uma resposta aritmética.

### Exemplo — o custo real de puxar a curva

São 8 tamanhos × 5 dobras = **40 treinos**, contra os 5 de um `cross_val_score`. Mas 40 treinos
não custam 8 vezes mais, porque **35 deles são menores**: o custo acompanha as **linhas**
treinadas, não a contagem de treinos.

In [ ]:
linhas_curva = (frac * tam[-1]).sum() * cv.get_n_splits()
linhas_cheio = tam[-1] * cv.get_n_splits()

print(f"linhas treinadas pela curva   : {linhas_curva:8.0f}")
print(f"linhas treinadas por 5 cheios : {linhas_cheio:8.0f}")
print(f"razão: {linhas_curva / linhas_cheio:.1f}x  (a contagem ingênua de treinos diria 8,0x)")
print("\nA soma das frações é o fator:", round(frac.sum(), 2))

### Exercício 6 — o tamanho suficiente

Calcule o **piso** da faixa do maior tamanho (média menos um desvio) e descubra o **menor**
tamanho de treino cuja média já fica acima desse piso. Imprima esse tamanho, a validação dele e
que fração da base ele representa.

Responda: com quantos pacientes essa base já entrega o desempenho do fim da curva?

In [ ]:
# TODO: piso = média do último ponto menos o desvio DESSE ponto
piso = ____

dentro = [(int(t), v) for t, v in zip(tam, m_va) if v >= piso]
suficiente, v_suf = dentro[0]

print(f"faixa do maior tamanho : {m_va[-1]:.4f} ± {s_va[-1]:.4f}  -> piso {piso:.4f}")
print(f"menor tamanho que alcança o piso: {suficiente} exemplos  (validação {v_suf:.4f})")
print(f"isso é {suficiente / tam[-1]:.0%} do treino disponível")

**Resposta:**

## Parte 7 — A curva fora do Pipeline

A curva refaz a validação cruzada 8 vezes. Se o pré-processamento estiver **fora** do Pipeline —
padronizando a base inteira antes de chamar a função — a padronização vê a validação de cada
dobra, em todos os 8 tamanhos. É o mesmo erro das Aulas 05 a 07, multiplicado.

### Exemplo — a matriz padronizada de uma vez só

In [ ]:
X_z = pd.DataFrame(StandardScaler().fit_transform(X), columns=feat)
print("padronizada na base INTEIRA — média e desvio de uma coluna:")
print(round(X_z[feat[0]].mean(), 6), round(X_z[feat[0]].std(), 4))

### Exercício 7 — o erro que não acende alarme

Gere a curva de aprendizado de uma `LogisticRegression` **crua** (sem Pipeline) sobre `X_z`, com
os mesmos `frac` e `cv`, e compare a validação ponto a ponto com a curva correta da Parte 1.

Responda: a diferença é grande? E, se é pequena, por que isso é um problema e não um alívio?

In [ ]:
# TODO: curva da LogisticRegression(max_iter=1000) sobre X_z, sem Pipeline
tam_z, tr_z, va_z = ____

for t, fora, dentro in zip(tam_z, va_z.mean(axis=1), m_va):
    print(f"{t:4d}   fora do Pipeline {fora:.4f}   dentro {dentro:.4f}   "
          f"diferença {fora - dentro:+.4f}")

**Resposta:**

---

**Parabéns!** Você construiu curvas de aprendizado para três modelos, leu as duas assinaturas,
transformou a inclinação final numa resposta de negócio, viu o primeiro ponto quebrar numa base
pequena, calculou o tamanho de amostra suficiente e o custo de descobri-lo, e mediu o vazamento
que não acende alarme.

Para levar ao projeto integrado: antes de pedir mais dados ao cliente, puxe a curva. Se a
validação achatou, o pedido não se sustenta — e se ela ainda sobe, você tem a figura que
justifica o orçamento.

Na próxima aula (03/11): **seleção e comparação de modelos, limiar de decisão e calibração**.